# Lab F — Contract Intelligence Semantic & Genie Ontology Layer

## Purpose

Create a governed business semantic layer above the Contract Intelligence Gold table so Genie can understand terms such as Vendor, Counterparty, Agreement, Commercial Exposure, Risk Rating, Renewal Attention, and Payment Exposure without relying only on raw column names.

This notebook does not create a separate ontology database. It creates modeled semantic context in Unity Catalog using a Metric View. Genie Ontology can use that modeled context together with inferred context from Genie Agents, queries, and other assets.

## Architecture

```text
Lab C
Bronze → Silver → Gold
                    │
                    ▼
Lab F
        Unity Catalog Metric View
                    │
          Dimensions + Measures
          Comments + Synonyms
                    │
                    ▼
           Genie Ontology Context
                    │
              ┌─────┴─────┐
              ▼           ▼
          Genie Agent   Genie One

Lab E Databricks App remains unchanged
and continues to use Gold + Silver directly.
```

## What this lab creates

`contract_intelligence_metrics`

with governed business dimensions, measures, display names, comments, and synonyms.

## Testing strategy

1. Metric View SQL test
2. Genie Agent test
3. Genie One test

The Databricks App does not automatically change because it queries Gold/Silver directly.


## Section 1 — Validate the Gold Foundation

In [0]:
# Discover the current catalog/schema and validate that Lab C created the Gold contract table.
row = spark.sql(
    "SELECT current_catalog() AS catalog_name, current_schema() AS schema_name"
).first()

CATALOG = row["catalog_name"]
SCHEMA = row["schema_name"]

GOLD_TABLE = f"{CATALOG}.{SCHEMA}.gold_enterprise_contracts"
METRIC_VIEW = f"{CATALOG}.{SCHEMA}.contract_intelligence_metrics"

if not spark.catalog.tableExists(GOLD_TABLE):
    raise RuntimeError(
        f"Run Lab C first. Missing required Gold table: {GOLD_TABLE}"
    )

print("Gold source:", GOLD_TABLE)
print("Semantic metric view:", METRIC_VIEW)

display(
    spark.table(GOLD_TABLE)
    .limit(5)
)


## Section 2 — Define the Contract Intelligence Business Ontology

The ontology is represented here as governed business concepts and mappings.

```text
Business language
      ↓
Semantic concept
      ↓
Metric View field / measure
      ↓
Gold implementation
```


In [0]:
# Define the business glossary that the Metric View will implement through fields, measures, comments, and synonyms.
business_glossary = [
    {
        "business_concept": "Supplier",
        "type": "Dimension",
        "definition": "External organization providing goods or services under a contract.",
        "synonyms": "Vendor, Counterparty, Service Provider",
        "implemented_as": "supplier_name",
    },
    {
        "business_concept": "Contract",
        "type": "Dimension",
        "definition": "Commercial agreement between the enterprise and a supplier.",
        "synonyms": "Agreement, Commercial Agreement",
        "implemented_as": "contract_id",
    },
    {
        "business_concept": "Contract Value",
        "type": "Field / Measure",
        "definition": "Commercial value of the contract in its recorded currency.",
        "synonyms": "Agreement Value, Commercial Value, Commercial Exposure",
        "implemented_as": "contract_value / total_contract_value",
    },
    {
        "business_concept": "Contract Risk",
        "type": "Dimension",
        "definition": "Current assessed risk level and risk category associated with a contract.",
        "synonyms": "Risk Rating, Risk Level, Contract Exposure",
        "implemented_as": "risk_severity / risk_type",
    },
    {
        "business_concept": "Renewal Review Due",
        "type": "Dimension / Measure",
        "definition": "Contract has entered its configured renewal-review window.",
        "synonyms": "Renewal Attention, Renewal Due, Renewal Exposure",
        "implemented_as": "renewal_review_due / renewal_review_due_count",
    },
    {
        "business_concept": "Upcoming Payment Exposure",
        "type": "Measure",
        "definition": "Value of next payments due within the next 90 days.",
        "synonyms": "Payment Exposure, Upcoming Payments, Near-Term Payments",
        "implemented_as": "upcoming_payment_exposure",
    },
    {
        "business_concept": "High-Risk Contract",
        "type": "Measure",
        "definition": "Contract whose risk severity is HIGH.",
        "synonyms": "Critical Contract, High Exposure Contract",
        "implemented_as": "high_risk_contract_count",
    },
    {
        "business_concept": "Open Obligation",
        "type": "Measure",
        "definition": "Outstanding contractual obligations recorded for active contracts.",
        "synonyms": "Outstanding Commitment, Open Commitment",
        "implemented_as": "total_open_obligations",
    },
]

glossary_df = spark.createDataFrame(business_glossary)
display(glossary_df)


## Section 3 — Define the Unity Catalog Metric View

The definition uses YAML version 1.1 with:
- `display_name`
- `comment`
- `synonyms`

These metadata fields are intended for downstream AI/BI and Genie experiences.


In [0]:
# Build the YAML 1.1 semantic definition for the Contract Intelligence Metric View.
metric_view_yaml = f"""
version: 1.1
comment: 'Governed Contract Intelligence semantic model for procurement, finance, legal, and operations analysis.'
source: {GOLD_TABLE}

fields:
  - name: contract_id
    expr: source.contract_id
    display_name: 'Contract'
    comment: 'Unique enterprise contract identifier.'
    synonyms: ['agreement', 'commercial agreement', 'contract number', 'agreement id']

  - name: supplier_id
    expr: source.supplier_id
    display_name: 'Supplier ID'
    comment: 'Unique identifier for the contracted supplier.'
    synonyms: ['vendor id', 'counterparty id']

  - name: supplier_name
    expr: source.supplier_name
    display_name: 'Supplier'
    comment: 'External organization providing goods or services under the contract.'
    synonyms: ['vendor', 'counterparty', 'service provider']

  - name: supplier_category
    expr: source.category
    display_name: 'Supplier Category'
    comment: 'Commercial category assigned to the supplier.'
    synonyms: ['vendor category', 'procurement category', 'category']

  - name: supplier_country
    expr: source.country
    display_name: 'Supplier Country'
    comment: 'Country associated with the supplier.'
    synonyms: ['vendor country', 'counterparty country']

  - name: supplier_tier
    expr: source.supplier_tier
    display_name: 'Supplier Tier'
    comment: 'Enterprise relationship tier assigned to the supplier.'
    synonyms: ['vendor tier', 'supplier segment', 'strategic supplier status']

  - name: contract_owner
    expr: source.contract_owner
    display_name: 'Contract Owner'
    comment: 'Enterprise function accountable for the contract.'
    synonyms: ['agreement owner', 'business owner', 'contract accountable owner']

  - name: contract_status
    expr: source.status
    display_name: 'Contract Status'
    comment: 'Current lifecycle status of the contract.'
    synonyms: ['agreement status', 'contract state']

  - name: contract_currency
    expr: source.currency
    display_name: 'Contract Currency'
    comment: 'Currency in which the contract value is recorded.'
    synonyms: ['agreement currency', 'commercial currency']

  - name: contract_value
    expr: source.contract_value
    display_name: 'Contract Value'
    comment: 'Commercial value of the individual contract.'
    synonyms: ['agreement value', 'commercial value', 'commercial exposure']

  - name: start_date
    expr: source.start_date
    display_name: 'Contract Start Date'
    comment: 'Effective start date of the contract.'
    synonyms: ['agreement start date', 'effective date']

  - name: end_date
    expr: source.end_date
    display_name: 'Contract End Date'
    comment: 'Current contractual expiry date.'
    synonyms: ['agreement end date', 'expiry date', 'expiration date']

  - name: days_to_expiry
    expr: source.days_to_expiry
    display_name: 'Days to Expiry'
    comment: 'Number of days from today until contract expiry.'
    synonyms: ['days remaining', 'time to expiry', 'time to expiration']

  - name: renewal_review_due
    expr: source.renewal_review_due
    display_name: 'Renewal Review Due'
    comment: 'True when the contract has entered its configured renewal-review window.'
    synonyms: ['renewal attention', 'renewal due', 'renewal exposure']

  - name: risk_type
    expr: source.risk_type
    display_name: 'Risk Type'
    comment: 'Primary category of identified contract risk.'
    synonyms: ['risk category', 'exposure type']

  - name: risk_severity
    expr: source.risk_severity
    display_name: 'Risk Severity'
    comment: 'Current assessed severity of contract risk.'
    synonyms: ['risk rating', 'risk level', 'contract risk']

  - name: next_milestone
    expr: source.next_milestone
    display_name: 'Next Milestone'
    comment: 'Next open contractual milestone.'
    synonyms: ['upcoming milestone', 'next deliverable']

  - name: next_milestone_date
    expr: source.next_milestone_date
    display_name: 'Next Milestone Date'
    comment: 'Target date of the next open milestone.'
    synonyms: ['milestone due date', 'deliverable due date']

  - name: next_payment_amount
    expr: source.next_payment_amount
    display_name: 'Next Payment Amount'
    comment: 'Value of the next scheduled payment.'
    synonyms: ['next payment', 'payment exposure', 'upcoming payment amount']

  - name: next_payment_due_date
    expr: source.next_payment_due_date
    display_name: 'Next Payment Due Date'
    comment: 'Due date of the next scheduled payment.'
    synonyms: ['payment due date', 'next payment date']

  - name: open_obligations
    expr: source.open_obligations
    display_name: 'Open Obligations'
    comment: 'Number of outstanding contract obligations.'
    synonyms: ['outstanding obligations', 'open commitments', 'outstanding commitments']

  - name: sla_target_pct
    expr: source.sla_target_pct
    display_name: 'SLA Target'
    comment: 'Contractual service-level target percentage.'
    synonyms: ['service level target', 'SLA requirement']

  - name: delay_penalty_pct_per_week
    expr: source.delay_penalty_pct_per_week
    display_name: 'Delay Penalty Per Week'
    comment: 'Contractual delay penalty percentage per week.'
    synonyms: ['delay penalty', 'late delivery penalty', 'penalty rate']

  - name: recommended_action
    expr: source.recommended_action
    display_name: 'Recommended Action'
    comment: 'Current governed recommendation for contract follow-up.'
    synonyms: ['recommended intervention', 'next action', 'recommended follow-up']

measures:
  - name: contract_count
    expr: COUNT(1)
    display_name: 'Contract Count'
    comment: 'Number of contracts in the selected business context.'
    synonyms: ['number of contracts', 'agreement count', 'number of agreements']

  - name: total_contract_value
    expr: SUM(source.contract_value)
    display_name: 'Total Contract Value'
    comment: 'Total commercial value of contracts in the selected context.'
    synonyms: ['commercial exposure', 'agreement value', 'portfolio value', 'contract exposure']

  - name: high_risk_contract_count
    expr: SUM(CASE WHEN source.risk_severity = 'HIGH' THEN 1 ELSE 0 END)
    display_name: 'High-Risk Contract Count'
    comment: 'Number of contracts whose risk severity is HIGH.'
    synonyms: ['high risk agreements', 'critical contracts', 'high exposure contracts']

  - name: total_open_obligations
    expr: SUM(source.open_obligations)
    display_name: 'Open Obligations'
    comment: 'Total outstanding contractual obligations.'
    synonyms: ['outstanding obligations', 'open commitments', 'outstanding commitments']

  - name: upcoming_payment_exposure
    expr: SUM(CASE WHEN source.next_payment_due_date BETWEEN CURRENT_DATE() AND DATE_ADD(CURRENT_DATE(), 90) THEN source.next_payment_amount ELSE 0 END)
    display_name: 'Upcoming Payment Exposure'
    comment: 'Value of next scheduled payments due within the next 90 days.'
    synonyms: ['payment exposure', 'upcoming payments', 'near-term payments']

  - name: renewal_review_due_count
    expr: SUM(CASE WHEN source.renewal_review_due = TRUE THEN 1 ELSE 0 END)
    display_name: 'Renewal Review Due Count'
    comment: 'Number of contracts currently requiring renewal review.'
    synonyms: ['renewals due', 'renewal attention', 'renewal exposure count']

  - name: average_days_to_expiry
    expr: AVG(source.days_to_expiry)
    display_name: 'Average Days to Expiry'
    comment: 'Average number of days remaining until contract expiry.'
    synonyms: ['average time to expiry', 'average days remaining']
"""

print(metric_view_yaml)


## Section 4 — Create the Metric View

The notebook attempts:

```sql
CREATE OR REPLACE VIEW ... WITH METRICS LANGUAGE YAML
```

If the workspace/runtime does not support the required Metric View features, the cell prints the exact error.


In [0]:
# Create the Unity Catalog Metric View and capture any feature/runtime limitation without hiding the error.
METRIC_VIEW_CREATED = False
METRIC_VIEW_ERROR = None

create_metric_view_sql = f"""
CREATE OR REPLACE VIEW {METRIC_VIEW}
WITH METRICS
LANGUAGE YAML
AS
$$
{metric_view_yaml}
$$
"""

try:
    spark.sql(create_metric_view_sql)
    METRIC_VIEW_CREATED = True
    print("Metric View created successfully:", METRIC_VIEW)

except Exception as metric_view_error:
    METRIC_VIEW_ERROR = str(metric_view_error)
    print("Metric View creation did not complete.")
    print(type(metric_view_error).__name__)
    print(METRIC_VIEW_ERROR[:2500])


## Section 5 — Inspect the Semantic Definition

In [0]:
# Inspect the stored Metric View definition and agent metadata when Metric View creation succeeds.
if METRIC_VIEW_CREATED:
    try:
        semantic_definition = spark.sql(
            f"DESCRIBE TABLE EXTENDED {METRIC_VIEW} AS JSON"
        )
        display(semantic_definition)

    except Exception as describe_error:
        print("Metric View exists, but JSON DESCRIBE was unavailable.")
        print(type(describe_error).__name__, str(describe_error)[:1000])

else:
    print("Skipping semantic-definition inspection because the Metric View was not created.")


## Section 6 — Test the Metric View Directly

Test:

> Which suppliers have the greatest commercial exposure and how many high-risk contracts do they have?


In [0]:
# Test governed supplier-level contract measures using the Metric View MEASURE() function.
if METRIC_VIEW_CREATED:
    supplier_exposure_query = f"""
    SELECT
        supplier_name,
        MEASURE(total_contract_value) AS total_contract_value,
        MEASURE(contract_count) AS contract_count,
        MEASURE(high_risk_contract_count) AS high_risk_contract_count
    FROM {METRIC_VIEW}
    GROUP BY supplier_name
    ORDER BY total_contract_value DESC
    """

    supplier_exposure_df = spark.sql(supplier_exposure_query)
    display(supplier_exposure_df)

else:
    print("Metric View test skipped.")


Second test:

> Which suppliers require renewal attention and what is their upcoming payment exposure?


In [0]:
# Test renewal and payment-exposure semantics using governed Metric View measures.
if METRIC_VIEW_CREATED:
    renewal_exposure_query = f"""
    SELECT
        supplier_name,
        MEASURE(renewal_review_due_count) AS renewal_review_due_count,
        MEASURE(upcoming_payment_exposure) AS upcoming_payment_exposure
    FROM {METRIC_VIEW}
    GROUP BY supplier_name
    HAVING MEASURE(renewal_review_due_count) > 0
    ORDER BY upcoming_payment_exposure DESC
    """

    renewal_exposure_df = spark.sql(renewal_exposure_query)
    display(renewal_exposure_df)

else:
    print("Metric View test skipped.")


## Section 7 — Test Contract-Level Semantic Fields

This supports questions such as:

> What action is recommended for agreement CTR-2026-002?

The word `agreement` is modeled as a synonym for Contract.


In [0]:
# Test contract-level semantic fields while keeping governed measures available for aggregation.
if METRIC_VIEW_CREATED:
    sample_contract = (
        spark.table(GOLD_TABLE)
        .orderBy("contract_id")
        .select("contract_id")
        .first()["contract_id"]
    )

    contract_semantic_query = f"""
    SELECT
        contract_id,
        supplier_name,
        risk_type,
        risk_severity,
        next_milestone,
        next_payment_amount,
        delay_penalty_pct_per_week,
        renewal_review_due,
        recommended_action
    FROM {METRIC_VIEW}
    WHERE contract_id = '{sample_contract}'
    GROUP BY ALL
    """

    print("Semantic contract test:", sample_contract)
    display(spark.sql(contract_semantic_query))

else:
    print("Metric View test skipped.")


## Section 8 — Semantic Phrase Mapping Test

This is a transparent check of the vocabulary intentionally modeled in the Metric View.


In [0]:
# Display the business phrases that should resolve to governed Metric View fields or measures in Genie.
semantic_phrase_tests = [
    ("vendor", "supplier_name", "Field synonym"),
    ("counterparty", "supplier_name", "Field synonym"),
    ("agreement", "contract_id", "Field synonym"),
    ("commercial exposure", "total_contract_value", "Measure synonym"),
    ("risk rating", "risk_severity", "Field synonym"),
    ("renewal attention", "renewal_review_due / renewal_review_due_count", "Field + measure synonym"),
    ("payment exposure", "upcoming_payment_exposure", "Measure synonym"),
    ("outstanding commitments", "total_open_obligations", "Measure synonym"),
]

semantic_phrase_df = spark.createDataFrame(
    semantic_phrase_tests,
    ["business_phrase", "expected_semantic_mapping", "mapping_type"]
)

display(semantic_phrase_df)


## Section 9 — Update the Existing Genie Agent

This step is completed in the Databricks UI after the notebook runs.

Go to:

**Genie Agents → Enterprise Contract Management → Configure → Sources**

Add:

```text
workspace.default.contract_intelligence_metrics
```

or the catalog/schema printed by this notebook.

### Recommended Agent instruction

```text
Use contract_intelligence_metrics as the preferred semantic source
for portfolio analytics.

Use governed Metric View measures for:
- Total Contract Value
- Contract Count
- High-Risk Contract Count
- Open Obligations
- Upcoming Payment Exposure
- Renewal Review Due Count

Business terminology:
- Vendor and Counterparty mean Supplier.
- Agreement means Contract.
- Commercial Exposure means Total Contract Value.
- Risk Rating and Risk Level mean Risk Severity.
- Renewal Attention means Renewal Review Due.
- Payment Exposure means Upcoming Payment Exposure.

Do not invent contract clauses.
For clause-level interpretation, direct the user to the
Contract Intelligence Databricks App.
```

For initial testing, keep both the existing Gold source and the Metric View. Once validated, you can make the Metric View the preferred analytical source.


## Section 10 — Genie Agent Before / After Test

Ask the same questions before and after adding the Metric View:

1. `Which vendors have the greatest commercial exposure?`
2. `How many high-risk agreements do we have?`
3. `Which counterparties need renewal attention?`
4. `Show payment exposure by vendor.`
5. `Which vendors have the greatest commercial exposure and high contract risk?`

For each answer:

- inspect generated SQL,
- confirm the Metric View is used where appropriate,
- confirm `MEASURE(...)` is used for governed measures,
- compare with the direct SQL tests in this notebook.


## Section 11 — Genie One Test

After updating and sharing the Genie Agent, test through Genie One:

```text
Which vendors have the greatest commercial exposure and high contract risk?
```

Expected flow:

```text
Business language
       ↓
Genie One
       ↓
Genie Ontology context
       ↓
Enterprise Contract Management Genie Agent
       ↓
contract_intelligence_metrics
       ↓
gold_enterprise_contracts
       ↓
Business answer
```

Verify:

- Vendor → Supplier
- Commercial Exposure → Total Contract Value
- High Contract Risk → Risk Severity
- governed Metric View measures are used
- the answer is consistent with direct SQL tests


## Section 12 — What Changes and What Does Not

| Component | Effect of Lab F |
|---|---|
| Gold table | Unchanged |
| Silver contract evidence | Unchanged |
| Lab D agents | Unchanged |
| Databricks App | Unchanged |
| Metric View | New |
| Genie semantic understanding | Enhanced |
| Genie Agent | Enhanced after adding Metric View source |
| Genie One | Uses improved business context through Genie |
| AI Search / RAG | Separate capability; unchanged |

The App does not change automatically because it still queries Gold and Silver directly.


## Section 13 — Lab F Status Summary

In [0]:
# Produce a concise Lab F status summary for troubleshooting and handoff.
import json

lab_f_summary = {
    "gold_source": GOLD_TABLE,
    "metric_view": METRIC_VIEW,
    "metric_view_created": METRIC_VIEW_CREATED,
    "metric_view_error": METRIC_VIEW_ERROR,
    "semantic_features": [
        "Business definitions",
        "Display names",
        "Field synonyms",
        "Measure synonyms",
        "Governed measures",
    ],
    "governed_measures": [
        "contract_count",
        "total_contract_value",
        "high_risk_contract_count",
        "total_open_obligations",
        "upcoming_payment_exposure",
        "renewal_review_due_count",
        "average_days_to_expiry",
    ],
    "downstream_target": [
        "Enterprise Contract Management Genie Agent",
        "Genie One",
    ],
    "unchanged_components": [
        "Lab C Gold/Silver pipeline",
        "Lab D agent workflow",
        "Lab E Databricks App",
    ],
}

print(json.dumps(lab_f_summary, indent=2))


## Lab F Completion Checklist

### Semantic model
- Gold foundation validated
- Contract Intelligence business glossary defined
- Metric View YAML 1.1 defined
- Business display names defined
- Business synonyms defined
- Governed measures defined
- Metric View creation attempted

### Direct validation
- Supplier commercial exposure test
- High-risk contract measure test
- Renewal-review test
- Payment-exposure test
- Contract-level field test
- Semantic phrase mapping documented

### Genie validation
- Add `contract_intelligence_metrics` to the Genie Agent
- Add semantic instructions
- Ask before/after business-language questions
- Inspect generated SQL
- Confirm governed `MEASURE(...)` usage

### Genie One validation
- Ask the same synonym-rich business question
- Confirm routing to the Contract Management Genie Agent
- Confirm terminology resolves to modeled semantics

## Final Architecture

```text
Lab C
Sources → Bronze → Silver → Gold

Lab F
Gold → Metric View → Genie Ontology Context
                         │
                         ├── Genie Agent
                         └── Genie One

Lab D
Gold + Silver → Agents → Evals → Human Approval → Action Queue

Lab E
Gold + Silver → Databricks App → Business Workflow
```
